# 03 – Predictive Modeling & Risk-Based Pricing

This notebook builds:

1. A **claim severity model** (regression) – predicts `TotalClaims` for policies with a claim.
2. A **claim probability model** (classification) – predicts `HasClaim`.
3. A simple **risk-based premium** calculation using the two models.


In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

from src.data.load_data import load_processed_data
from src.features.build_features import add_core_features, get_default_feature_lists

pd.set_option("display.max_columns", 100)

## 1. Load the processed dataset and add features

In [ ]:
df = load_processed_data()
df = add_core_features(df)
df.head()

## 2. Define numeric and categorical feature lists

In [ ]:
numeric_features, categorical_features = get_default_feature_lists(df)

numeric_features, categorical_features

## 3. Claim severity model (regression)

In [ ]:
# Use only rows with a claim
df_sev = df[df.get("HasClaim", 0) == 1].copy()

X_sev = df_sev[numeric_features + categorical_features]
y_sev = df_sev["TotalClaims"]

X_train_sev, X_test_sev, y_train_sev, y_test_sev = train_test_split(
    X_sev, y_sev, test_size=0.2, random_state=42
)

numeric_transformer = SimpleImputer(strategy="median")
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

rf_reg = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
)

sev_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", rf_reg),
    ]
)

sev_model.fit(X_train_sev, y_train_sev)
y_pred_sev = sev_model.predict(X_test_sev)

rmse = mean_squared_error(y_test_sev, y_pred_sev, squared=False)
r2 = r2_score(y_test_sev, y_pred_sev)

print(f"Severity model RMSE: {rmse:.2f}")
print(f"Severity model R^2:  {r2:.3f}")

## 4. Claim probability model (classification)

In [ ]:
if "HasClaim" not in df.columns:
    raise ValueError("Expected 'HasClaim' column in data.")

X_cls = df[numeric_features + categorical_features]
y_cls = df["HasClaim"]

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

rf_cls = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    n_jobs=-1,
)

cls_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", rf_cls),
    ]
)

cls_model.fit(X_train_cls, y_train_cls)
y_proba_cls = cls_model.predict_proba(X_test_cls)[:, 1]

auc = roc_auc_score(y_test_cls, y_proba_cls)
print(f"Classification model AUC: {auc:.3f}")

## 5. Conceptual risk-based premium

In [ ]:
# Example: compute expected loss for each record in the classification test set
sev_pred_for_cls = sev_model.predict(X_test_cls)  # reuse severity model
expected_loss = y_proba_cls * sev_pred_for_cls

# Add a simple expense + profit loading, e.g. +15%
loading_factor = 1.15
premium_suggested = expected_loss * loading_factor

results = pd.DataFrame({
    "ExpectedLoss": expected_loss,
    "SuggestedPremium": premium_suggested,
})

results.head()

## 6. Next steps

- Compare `SuggestedPremium` to actual premiums in the data (e.g. `CalculatedPremiumPerTerm`).
- Analyse which segments are under-priced or over-priced.
- Use SHAP/LIME in a separate notebook or in this one to understand
  the main features driving the models.
